In [ ]:
%load_ext autoreload
%autoreload 2
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

ProjDIR = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
sys.path.insert(1, f'{ProjDIR}/src/')
from ASD_Circuits import *
from plot import *

HGNC, ENSID2Entrez, GeneSymbol2Entrez, Entrez2Symbol = LoadGeneINFO()

In [ ]:
# Load config and expression matrix
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

STR_BiasMat = pd.read_parquet(f"../{config['analysis_types']['STR_ISH']['expr_matrix']}")
STR_Anno = STR2Region()

In [ ]:
# Load connectivity matrices and null CCS
ScoreMatDir = os.path.join(ProjDIR, "dat/allen-mouse-conn/ConnectomeScoringMat")
IpsiInfoMat = pd.read_csv(os.path.join(ScoreMatDir, "InfoMat.Ipsi.csv"), index_col=0)

DIR = os.path.join(ProjDIR, "dat/allen-mouse-conn/RankScores")
Cont_Distance = np.load(os.path.join(DIR, "RankScore.Ipsi.Cont.npy"))
topNs = np.arange(200, 5, -1)

In [ ]:
# Load DDD mutation file
df = pd.read_excel("/home/jw3514/Work/data/DDD/41586_2020_2832_MOESM4_ESM.xlsx")
df = df.sort_values("denovoWEST_p_full")
hc_df = df[df["denovoWEST_p_full"]<=0.05/18762]
entrez_ids = [int(GeneSymbol2Entrez.get(x, -1)) for x in hc_df["symbol"].values]
hc_df["EntrezID"] = entrez_ids
hc_df.shape

In [ ]:
# Load ASD bias and gene weights
#Spark_ASD_STR_Bias = pd.read_csv("../dat/Unionize_bias/Spark_Meta_EWS.Z2.bias.FDR.SubSampleSib.csv", index_col=0)
Spark_ASD_STR_Bias = pd.read_csv("/mnt/data0/home_backup/Work/ASD_Circuits/dat/Unionize_bias/Spark_Meta_EWS.Z2.bias.csv", index_col=0)
Spark_ASD_STR_Bias["Region"] = Spark_ASD_STR_Bias["REGION"]
ASD_GW = Fil2Dict(os.path.join(ProjDIR, "dat/Genetics/GeneWeights/ASD_All.gw"))
ASD_GENES = list(ASD_GW.keys())

# Load DDD bias (recomputed from config gene weights)
#DDD_GW = Fil2Dict(config["gene_sets"]["DDD_293"]["geneweights"])
DDD_MutFil = hc_df
DDD_GW = Aggregate_Gene_Weights_NDD(DDD_MutFil)
DDD_STR_Bias = MouseSTR_AvgZ_Weighted(STR_BiasMat, DDD_GW)
DDD_STR_Bias["Region"] = [STR_Anno.get(s, "Unknown") for s in DDD_STR_Bias.index]

# DDD excluding ASD genes (recompute from full DDD minus ASD_All genes)
DDD_GW_filt_ASD = {k: v for k, v in DDD_GW.items() if k not in ASD_GENES}
print(f"DDD genes: {len(DDD_GW)}, after excluding ASD ({len(ASD_GENES)} genes): {len(DDD_GW_filt_ASD)}")
DDD_rmASD_STR_Bias = MouseSTR_AvgZ_Weighted(STR_BiasMat, DDD_GW_filt_ASD)
DDD_rmASD_STR_Bias["Region"] = [STR_Anno.get(s, "Unknown") for s in DDD_rmASD_STR_Bias.index]

In [ ]:
# Load circuit structures (canonical pipeline output)
ASD_CircuitsSet = pd.read_csv("../results/STR_ISH/ASD.SA.Circuits.Size46.csv", index_col="idx")
Circuit_STRs = ASD_CircuitsSet.loc[3, "STRs"].split(";")

# Section 1: DDD vs ASD -- Structure Level

## 1.1 All DDD genes: CCS plot

In [ ]:
score_DDD_all = calculate_circuit_scores(DDD_STR_Bias, IpsiInfoMat, sort_by="EFFECT")

plt.style.use('seaborn-v0_8-whitegrid')
fig, ax1 = plt.subplots(1, 1, dpi=480, figsize=(12, 6), facecolor='none')
fig.patch.set_alpha(0)
ax1.patch.set_alpha(0)

BarLen = 34.1
ax1.plot(topNs, score_DDD_all, color='#1f77b4', marker="o", markersize=5, lw=1,
         ls="dashed", label="DD", alpha=0.5)

cont = np.median(Cont_Distance, axis=0)
lower = np.percentile(Cont_Distance, 50 - BarLen, axis=0)
upper = np.percentile(Cont_Distance, 50 + BarLen, axis=0)
ax1.errorbar(topNs, cont, color="grey", marker="o", markersize=1.5, lw=1,
             yerr=(cont - lower, upper - cont), ls="dashed", label="Siblings")
ax1.set_xlabel("Structure Rank\n", fontsize=17)
ax1.set_ylabel("Circuit Connectivity Score", fontsize=15)
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_xlim(0, 121)
ax1.legend(fontsize=13, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()

## 1.2 DDD vs ASD correlation

In [ ]:
merged_data = merge_bias_datasets(Spark_ASD_STR_Bias, DDD_STR_Bias, suffixes=('_ASD', '_DD'))
plot_structure_bias_comparison(merged_data, suffixes=('_ASD', '_DD'), metric="EFFECT")

## 1.3 DDD (exclude ASD genes) vs ASD correlation

In [ ]:
merged_data2 = merge_bias_datasets(Spark_ASD_STR_Bias, DDD_rmASD_STR_Bias, suffixes=('_ASD', '_DD_ExcludeASD'))
#merged_data2 = merge_bias_datasets(Spark_ASD_STR_Bias, DDD_STR_Bias, suffixes=('_ASD', '_DD_ExcludeASD'))
plot_structure_bias_comparison(merged_data2, suffixes=('_ASD', '_DD_ExcludeASD'), metric='EFFECT')

## 1.4 Residual structures with bootstrap CI

In [ ]:
DDD_CI_path = "../results/Bootstrap_bias/DDD_ExomeWide/Residual_CI/DDD_ExomeWide.Residual_CI_95.csv"
DDD_residual_ci_df = pd.read_csv(DDD_CI_path, index_col=0)
merged_data_eval = merged_data2[merged_data2.index.isin(Circuit_STRs)]
top_diff_ci = plot_top_residual_structures_with_CI(merged_data_eval, DDD_residual_ci_df, top_n=20, top_threshold=40,
                                                   name1="ASD", name2="DD_ExcludeASD", figsize=(6, 8))

# Section 2: Constraint Gene Analysis (Structure Level)
Cell type DDD/Constraint analysis is in `notebooks_mouse_sc/08.DDD_Constraint_CellType`.

In [ ]:
# Load gnomAD v4 constraint data
gnomad4 = pd.read_csv(os.path.join(ProjDIR, "dat/Genetics/gnomad.v4.0.constraint_metrics.tsv"), sep="\t")
gnomad4 = gnomad4[(gnomad4["transcript"].str.contains('ENST'))]
gnomad4 = gnomad4[gnomad4["mane_select"] == True]
for i, row in gnomad4.iterrows():
    gnomad4.loc[i, "Entrez"] = int(GeneSymbol2Entrez.get(row["gene"], 0))

## 2.1 pLI >= 0.99 analysis

In [ ]:
gnomad4_top_PLI = gnomad4[gnomad4["lof.pLI"] > 0.99]
print(f"pLI>=0.99 genes: {gnomad4_top_PLI.shape[0]}")
constraint_gw_top_PLI = dict(zip(gnomad4_top_PLI["Entrez"], [1] * len(gnomad4_top_PLI)))
Dict2Fil(constraint_gw_top_PLI, os.path.join(ProjDIR, "dat/Genetics/GeneWeights/constraint_top_decile_PLI.gw"))

constraint_top_PLI_STR_Bias = MouseSTR_AvgZ_Weighted(STR_BiasMat, constraint_gw_top_PLI)
constraint_top_PLI_STR_Bias["Region"] = [STR_Anno.get(s, "Unknown") for s in constraint_top_PLI_STR_Bias.index]

# Structure bias comparisons
merged_data_ASD_Constraint_PLI = merge_bias_datasets(Spark_ASD_STR_Bias, constraint_top_PLI_STR_Bias, suffixes=('_ASD', '_Constrained'))
plot_structure_bias_comparison(merged_data_ASD_Constraint_PLI, suffixes=('_ASD', '_Constrained'), metric="EFFECT", show_region_legend=True)

merged_data_DDD_Constraint_PLI = merge_bias_datasets(DDD_STR_Bias, constraint_top_PLI_STR_Bias, suffixes=('_DD', '_Constrained'))
plot_structure_bias_comparison(merged_data_DDD_Constraint_PLI, suffixes=('_DD', '_Constrained'), metric="EFFECT")

In [ ]:
# CCS for pLI constraint genes
score_Constraint_PLI = calculate_circuit_scores(constraint_top_PLI_STR_Bias, IpsiInfoMat, sort_by="EFFECT")

In [ ]:
# Residual: ASD vs Constrained (pLI)
merged_data_eval = merged_data_ASD_Constraint_PLI[merged_data_ASD_Constraint_PLI.index.isin(Circuit_STRs)]
_ = plot_top_residual_structures_with_CI(merged_data_eval, top_n=20, top_threshold=40,
                                         name1="ASD", name2="Constrained", figsize=(6, 6))

# Residual: DDD vs Constrained (pLI)
merged_data_eval = merged_data_DDD_Constraint_PLI[merged_data_DDD_Constraint_PLI.index.isin(Circuit_STRs)]
_ = plot_top_residual_structures_with_CI(merged_data_eval, top_n=20, top_threshold=40,
                                         name1="DD", name2="Constrained", figsize=(6, 8))

## 2.2 LOEUF top 25% analysis

In [ ]:
bottom_25_percent_threshold = gnomad4["lof.oe_ci.upper"].quantile(0.25)
gnomad4_bottom25 = gnomad4[gnomad4["lof.oe_ci.upper"] <= bottom_25_percent_threshold]
columns_to_keep_g4 = ["Entrez", "gene", "lof.pLI", "lof.z_score", "lof.oe_ci.upper"]
gnomad4_bottom25 = gnomad4_bottom25[columns_to_keep_g4].copy()
gnomad4_bottom25["Entrez"] = gnomad4_bottom25["Entrez"].astype(int)
gnomad4_bottom25 = gnomad4_bottom25[gnomad4_bottom25["Entrez"] != 0]
gnomad4_bottom25 = gnomad4_bottom25.sort_values(by="lof.oe_ci.upper", ascending=True)
print(f"LOEUF top 25% genes: {gnomad4_bottom25.shape[0]}")

constraint_gw_top_LOEUF25 = dict(zip(gnomad4_bottom25["Entrez"], [1] * len(gnomad4_bottom25)))
Dict2Fil(constraint_gw_top_LOEUF25, os.path.join(ProjDIR, "dat/Genetics/GeneWeights/constraint_top25_LOEUF.gw"))

constraint_top_LOEUF25_STR_Bias = MouseSTR_AvgZ_Weighted(STR_BiasMat, constraint_gw_top_LOEUF25)
constraint_top_LOEUF25_STR_Bias["Region"] = [STR_Anno.get(s, "Unknown") for s in constraint_top_LOEUF25_STR_Bias.index]

# Compare with ASD
merged_data_ASD_Constraint_LOEUF25 = merge_bias_datasets(Spark_ASD_STR_Bias, constraint_top_LOEUF25_STR_Bias, suffixes=('_ASD', '_Constrained'))
plot_structure_bias_comparison(merged_data_ASD_Constraint_LOEUF25, suffixes=('_ASD', '_Constrained'), metric="EFFECT")

# Compare with DDD (exclude ASD)
merged_data_DDD_Constraint_LOEUF25 = merge_bias_datasets(DDD_rmASD_STR_Bias, constraint_top_LOEUF25_STR_Bias, suffixes=('_DD (exclude ASD)', '_Constrained'))
plot_structure_bias_comparison(merged_data_DDD_Constraint_LOEUF25, suffixes=('_DD (exclude ASD)', '_Constrained'), metric="EFFECT")

In [ ]:
# Residual: ASD vs Constrained (LOEUF top 25%)
merged_data_eval_LOEUF25 = merged_data_ASD_Constraint_LOEUF25[merged_data_ASD_Constraint_LOEUF25.index.isin(Circuit_STRs)]
_ = plot_top_residual_structures_with_CI(merged_data_eval_LOEUF25, top_n=20, top_threshold=40,
                                         name1="ASD", name2="Constrained", figsize=(6, 6))

In [ ]:
# Load LOEUF25 bootstrap CI and plot with error bars
LOEUF25_CI_path = "../results/Bootstrap_bias/LOEUF25/Residual_CI/LOEUF25.Residual_CI_95.csv"
LOEUF25_residual_ci_df = pd.read_csv(LOEUF25_CI_path, index_col=0)
_ = plot_top_residual_structures_with_CI(merged_data_eval_LOEUF25, LOEUF25_residual_ci_df, top_n=20, top_threshold=40,
                                         name1="ASD", name2="Constrained", figsize=(6, 8))

## 2.3 CCS comparison: DD (excl ASD) vs Constrained

In [ ]:
# Calculate all circuit scores
score_ASD = calculate_circuit_scores(Spark_ASD_STR_Bias, IpsiInfoMat, sort_by="EFFECT")
score_DDD = calculate_circuit_scores(DDD_STR_Bias, IpsiInfoMat, sort_by="EFFECT")
score_DDD_rmASD = calculate_circuit_scores(DDD_rmASD_STR_Bias, IpsiInfoMat, sort_by="EFFECT")
score_Constraint_LOEUF25 = calculate_circuit_scores(constraint_top_LOEUF25_STR_Bias, IpsiInfoMat, sort_by="EFFECT")

In [ ]:
# CCS plot: DD (excl ASD) vs Constrained (pLI)
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax1 = plt.subplots(1, 1, dpi=480, figsize=(10, 6), facecolor='none')
fig.patch.set_alpha(0)
ax1.patch.set_alpha(0)

BarLen = 34.1
cont = np.median(Cont_Distance, axis=0)
lower = np.percentile(Cont_Distance, 50 - BarLen, axis=0)
upper = np.percentile(Cont_Distance, 50 + BarLen, axis=0)

ax1.plot(topNs, score_DDD_rmASD, color="#ff7f0e", marker="o", markersize=5, lw=1,
         ls="dashed", label="DD (exclude ASD)", alpha=0.9)
ax1.plot(topNs, score_Constraint_PLI, color="#2ca02c", marker="o", markersize=5, lw=1,
         ls="dashed", label="Constrained Genes (pLI>=0.99)", alpha=0.9)
ax1.errorbar(topNs, cont, color="grey", marker="o", markersize=1.5, lw=1,
             yerr=(cont - lower, upper - cont), ls="dashed", label="Siblings")
ax1.set_xlabel("Structure Rank\n", fontsize=17)
ax1.set_ylabel("Circuit Connectivity Score", fontsize=15)
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_xlim(0, 121)
ax1.legend(fontsize=13, loc='upper right', frameon=True)
plt.tight_layout()

In [ ]:
# CCS: DD (excl ASD) vs Constrained (LOEUF top 25%)
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax1 = plt.subplots(1, 1, dpi=480, figsize=(10, 6), facecolor='none')
fig.patch.set_alpha(0)
ax1.patch.set_alpha(0)

ax1.plot(topNs, score_DDD_rmASD, color="#ff7f0e", marker="o", markersize=5, lw=1,
         ls="dashed", label="DD (exclude ASD)", alpha=0.9)
ax1.plot(topNs, score_Constraint_LOEUF25, color="#2ca02c", marker="o", markersize=5, lw=1,
         ls="dashed", label="Constrained Genes (LOEUF top 25%)", alpha=0.9)
ax1.errorbar(topNs, cont, color="grey", marker="o", markersize=1.5, lw=1,
             yerr=(cont - lower, upper - cont), ls="dashed", label="Siblings")
ax1.set_xlabel("Structure Rank\n", fontsize=17)
ax1.set_ylabel("Circuit Connectivity Score", fontsize=15)
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_xlim(0, 121)
ax1.legend(fontsize=13, loc='upper right', frameon=True)
plt.tight_layout()

## 2.4 pLI vs LOEUF comparison

In [ ]:
# Gene overlap
pLI_genes = set(constraint_gw_top_PLI.keys())
LOEUF25_genes = set(constraint_gw_top_LOEUF25.keys())

print(f"pLI>=0.99 genes: {len(pLI_genes)}")
print(f"LOEUF top 25% genes: {len(LOEUF25_genes)}")
print(f"Overlap: {len(pLI_genes & LOEUF25_genes)}")
print(f"Only in pLI: {len(pLI_genes - LOEUF25_genes)}")
print(f"Only in LOEUF top 25%: {len(LOEUF25_genes - pLI_genes)}")
print(f"Jaccard index: {len(pLI_genes & LOEUF25_genes) / len(pLI_genes | LOEUF25_genes):.3f}")

In [ ]:
# Direct structure bias correlation: pLI vs LOEUF
merged_data_pLI_LOEUF25 = merge_bias_datasets(constraint_top_PLI_STR_Bias, constraint_top_LOEUF25_STR_Bias, suffixes=('_pLI', '_LOEUF25'))
plot_structure_bias_comparison(merged_data_pLI_LOEUF25, suffixes=('_pLI', '_LOEUF25'), metric="EFFECT")

In [ ]:
# Summary structure correlations
print("=" * 60)
print("Structure Bias Correlations")
print("=" * 60)

corr_pLI_ASD, pval_pLI_ASD = pearsonr(merged_data_ASD_Constraint_PLI["EFFECT_ASD"], merged_data_ASD_Constraint_PLI["EFFECT_Constrained"])
corr_pLI_DDD, pval_pLI_DDD = pearsonr(merged_data_DDD_Constraint_PLI["EFFECT_DD"], merged_data_DDD_Constraint_PLI["EFFECT_Constrained"])
corr_LOEUF25_ASD, pval_LOEUF25_ASD = pearsonr(merged_data_ASD_Constraint_LOEUF25["EFFECT_ASD"], merged_data_ASD_Constraint_LOEUF25["EFFECT_Constrained"])
corr_LOEUF25_DDD, pval_LOEUF25_DDD = pearsonr(merged_data_DDD_Constraint_LOEUF25["EFFECT_DD (exclude ASD)"], merged_data_DDD_Constraint_LOEUF25["EFFECT_Constrained"])

print(f"\npLI>=0.99 ({len(pLI_genes)} genes):")
print(f"  ASD correlation:  r = {corr_pLI_ASD:.3f}, p = {pval_pLI_ASD:.2e}")
print(f"  DDD correlation:  r = {corr_pLI_DDD:.3f}, p = {pval_pLI_DDD:.2e}")
print(f"\nLOEUF top 25% ({len(LOEUF25_genes)} genes):")
print(f"  ASD correlation:  r = {corr_LOEUF25_ASD:.3f}, p = {pval_LOEUF25_ASD:.2e}")
print(f"  DDD correlation:  r = {corr_LOEUF25_DDD:.3f}, p = {pval_LOEUF25_DDD:.2e}")
print("=" * 60)

In [ ]:
# Side-by-side CCS: pLI vs LOEUF top 25%
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, dpi=480, figsize=(18, 6), facecolor='none')
fig.patch.set_alpha(0)
for ax in [ax1, ax2]:
    ax.patch.set_alpha(0)

ASD_color, DDD_color, rmASD_color = "#d62728", "#1f77b4", "#ff7f0e"
Constraint_color, siblings_color = "#2ca02c", "grey"

# Panel 1: pLI
ax1.plot(topNs, score_DDD, color=DDD_color, marker="o", markersize=5, lw=1, ls="dashed", label="DD", alpha=0.9)
ax1.plot(topNs, score_DDD_rmASD, color=rmASD_color, marker="o", markersize=5, lw=1, ls="dashed", label="DD (exclude ASD)", alpha=0.9)
ax1.plot(topNs, score_Constraint_PLI, color=Constraint_color, marker="o", markersize=5, lw=1, ls="dashed", label="Constrained (pLI>=0.99)", alpha=0.9)
ax1.errorbar(topNs, cont, color=siblings_color, marker="o", markersize=1.5, lw=1,
             yerr=(cont - lower, upper - cont), ls="dashed", label="Siblings")
ax1.set_xlabel("Structure Rank", fontsize=15)
ax1.set_ylabel("Circuit Connectivity Score", fontsize=15)
ax1.set_title("pLI>=0.99", fontsize=16, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.7)
ax1.set_xlim(0, 121)
ax1.legend(fontsize=11, loc='upper right', frameon=True)

# Panel 2: LOEUF top 25%
ax2.plot(topNs, score_DDD, color=DDD_color, marker="o", markersize=5, lw=1, ls="dashed", label="DD", alpha=0.9)
ax2.plot(topNs, score_DDD_rmASD, color=rmASD_color, marker="o", markersize=5, lw=1, ls="dashed", label="DD (exclude ASD)", alpha=0.9)
ax2.plot(topNs, score_Constraint_LOEUF25, color=Constraint_color, marker="o", markersize=5, lw=1, ls="dashed", label="Constrained (LOEUF top 25%)", alpha=0.9)
ax2.errorbar(topNs, cont, color=siblings_color, marker="o", markersize=1.5, lw=1,
             yerr=(cont - lower, upper - cont), ls="dashed", label="Siblings")
ax2.set_xlabel("Structure Rank", fontsize=15)
ax2.set_ylabel("Circuit Connectivity Score", fontsize=15)
ax2.set_title("LOEUF top 25%", fontsize=16, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.7)
ax2.set_xlim(0, 121)
ax2.legend(fontsize=11, loc='upper right', frameon=True)

plt.tight_layout()

# Section 3: Constraint Decile Analysis

Correlation between ASD/DDD structure bias and constraint genes across all 10 deciles of LOEUF.
- **Decile 1**: Most constrained (lowest LOEUF)
- **Decile 10**: Least constrained (highest LOEUF)

## 3.1 Decile correlation analysis (with p-values)

In [ ]:
decile_results_pval = []

for decile_num in range(1, 11):
    lower_quantile = (decile_num - 1) / 10
    upper_quantile = decile_num / 10
    lower_threshold = gnomad4["lof.oe_ci.upper"].quantile(lower_quantile)
    upper_threshold = gnomad4["lof.oe_ci.upper"].quantile(upper_quantile)

    gnomad4_decile = gnomad4[
        (gnomad4["lof.oe_ci.upper"] > lower_threshold) &
        (gnomad4["lof.oe_ci.upper"] <= upper_threshold)
    ]
    columns_to_keep = ["Entrez", "gene", "lof.pLI", "lof.z_score", "lof.oe_ci.upper"]
    gnomad4_decile = gnomad4_decile[columns_to_keep].copy()
    gnomad4_decile["Entrez"] = gnomad4_decile["Entrez"].astype(int)
    gnomad4_decile = gnomad4_decile[gnomad4_decile["Entrez"] != 0]

    constraint_gw_decile = dict(zip(gnomad4_decile["Entrez"], 1 / gnomad4_decile["lof.oe_ci.upper"]))
    constraint_STR_Bias_decile = MouseSTR_AvgZ_Weighted(STR_BiasMat, constraint_gw_decile)
    constraint_STR_Bias_decile["Region"] = [STR_Anno.get(s, "Unknown") for s in constraint_STR_Bias_decile.index]

    merged_ASD = merge_bias_datasets(Spark_ASD_STR_Bias, constraint_STR_Bias_decile, suffixes=('_ASD', '_Constrained'))
    corr_ASD, pval_ASD = pearsonr(merged_ASD["EFFECT_ASD"], merged_ASD["EFFECT_Constrained"])

    merged_DDD = merge_bias_datasets(DDD_rmASD_STR_Bias, constraint_STR_Bias_decile, suffixes=('_DD', '_Constrained'))
    corr_DDD, pval_DDD = pearsonr(merged_DDD["EFFECT_DD"], merged_DDD["EFFECT_Constrained"])

    decile_results_pval.append({
        'Decile': decile_num,
        'N_genes': len(gnomad4_decile),
        'LOEUF_mean': gnomad4_decile["lof.oe_ci.upper"].mean(),
        'Correlation_ASD': corr_ASD,
        'P_value_ASD': pval_ASD,
        'Correlation_DDD': corr_DDD,
        'P_value_DDD': pval_DDD,
        'Sig_ASD': '***' if pval_ASD < 0.001 else '**' if pval_ASD < 0.01 else '*' if pval_ASD < 0.05 else 'ns',
        'Sig_DDD': '***' if pval_DDD < 0.001 else '**' if pval_DDD < 0.01 else '*' if pval_DDD < 0.05 else 'ns'
    })
    print(f"Decile {decile_num}: N={len(gnomad4_decile)}, LOEUF=[{lower_threshold:.3f}, {upper_threshold:.3f}], "
          f"Corr_ASD={corr_ASD:.3f} ({pval_ASD:.2e}), Corr_DDD={corr_DDD:.3f} ({pval_DDD:.2e})")

decile_results_df = pd.DataFrame(decile_results_pval)
decile_results_df

## 3.2 Decile visualizations

In [ ]:
# Line plot: Correlation vs Decile / Mean LOEUF
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=240)

ax1 = axes[0]
ax1.plot(decile_results_df['Decile'], decile_results_df['Correlation_ASD'],
         marker='o', markersize=8, linewidth=2, label='ASD vs Constrained', color='#1f77b4')
ax1.plot(decile_results_df['Decile'], decile_results_df['Correlation_DDD'],
         marker='s', markersize=8, linewidth=2, label='DD (excl. ASD) vs Constrained', color='#ff7f0e')
ax1.set_xlabel('Constrained Decile\n(1=Most Constrained, 10=Least Constrained)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Correlation with Structure Bias', fontsize=14, fontweight='bold')
ax1.set_title('Correlation vs Constrained Decile', fontsize=16, fontweight='bold', pad=20)
ax1.legend(fontsize=12, loc='best')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_xticks(range(1, 11))
ax1.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

ax2 = axes[1]
ax2.plot(decile_results_df['LOEUF_mean'], decile_results_df['Correlation_ASD'],
         marker='o', markersize=8, linewidth=2, label='ASD vs Constrained', color='#1f77b4')
ax2.plot(decile_results_df['LOEUF_mean'], decile_results_df['Correlation_DDD'],
         marker='s', markersize=8, linewidth=2, label='DD (excl. ASD) vs Constrained', color='#ff7f0e')
ax2.set_xlabel('Mean LOEUF\n(Lower = More Constrained)', fontsize=14, fontweight='bold')
ax2.set_ylabel('Correlation with Structure Bias', fontsize=14, fontweight='bold')
ax2.set_title('Correlation vs Mean LOEUF', fontsize=16, fontweight='bold', pad=20)
ax2.legend(fontsize=12, loc='best')
ax2.grid(True, alpha=0.3, linestyle='--')
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Bar plot with significance annotations
fig, ax = plt.subplots(figsize=(14, 7), dpi=240)
x = np.arange(len(decile_results_df))
width = 0.35

bars1 = ax.bar(x - width / 2, decile_results_df['Correlation_ASD'], width,
               label='ASD vs Constrained', color='#1f77b4', alpha=0.8, edgecolor='black', linewidth=1)
bars2 = ax.bar(x + width / 2, decile_results_df['Correlation_DDD'], width,
               label='DD (excl. ASD) vs Constrained', color='#ff7f0e', alpha=0.8, edgecolor='black', linewidth=1)

for i, (idx, row) in enumerate(decile_results_df.iterrows()):
    if row['Sig_ASD'] != 'ns':
        y_pos = row['Correlation_ASD'] + (0.02 if row['Correlation_ASD'] > 0 else -0.05)
        va = 'bottom' if row['Correlation_ASD'] > 0 else 'top'
        ax.text(i - width / 2, y_pos, row['Sig_ASD'], ha='center', va=va, fontsize=10, fontweight='bold')
    if row['Sig_DDD'] != 'ns':
        y_pos = row['Correlation_DDD'] + (0.02 if row['Correlation_DDD'] > 0 else -0.05)
        va = 'bottom' if row['Correlation_DDD'] > 0 else 'top'
        ax.text(i + width / 2, y_pos, row['Sig_DDD'], ha='center', va=va, fontsize=10, fontweight='bold')

ax.set_xlabel('Constrained Decile (1=Most Constrained, 10=Least Constrained)', fontsize=14, fontweight='bold')
ax.set_ylabel('Correlation with Structure Bias', fontsize=14, fontweight='bold')
ax.set_title('Structure Bias Correlation Across Constrained Deciles\n(*** p<0.001, ** p<0.01, * p<0.05)',
             fontsize=16, fontweight='bold', pad=20)
ax.set_xticks(x)
ax.set_xticklabels([f'{i}\n({row["LOEUF_mean"]:.2f})' for i, row in decile_results_df.iterrows()], fontsize=10)
ax.legend(fontsize=12, loc='best')
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=1)
ax.grid(True, alpha=0.3, linestyle='--', axis='y')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()